# Tuần 2 — PhoBERT Multi-task Finetune (Anti-OOM)
**ABSA VLSP 2018 Hotel | NLP Course — HUST**

Notebook này dựa trên `week2_phobert_training_version1.ipynb` và chỉ sửa cấu hình để hạn chế OOM CUDA:
- `max_seq_len = 256`
- `batch_size = 4`
- `grad_accumulation_steps = 8`

> Chạy theo thứ tự từ Cell 1 đến Cell 9.

In [1]:
# ============================================================
# Cell 1 — Check GPU & Install dependencies
# ============================================================
import torch

if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu}")
    print(f"   VRAM: {vram:.1f} GB")
    if vram < 14:
        print("⚠️  VRAM < 14GB — nếu OOM thì giảm batch_size về 4 trong constants.py")
else:
    raise RuntimeError("❌ Không có GPU! Kaggle: Settings → Accelerator → GPU T4 x2")

torch.cuda.empty_cache()
print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")

# Install packages
!pip install -q transformers==4.38.0 underthesea py_vncorenlp tabulate tqdm scikit-learn sentencepiece
print("✅ Dependencies installed")

✅ GPU: Tesla T4
   VRAM: 15.6 GB
PyTorch: 2.10.0+cu128 | CUDA: 12.8
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 71.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.38.0 which is incompatible.
✅ Dependencies installed


In [2]:
# ============================================================
# Cell 2 — Clone repo từ GitHub & Setup working directory
# ============================================================
import os, sys

REPO_URL    = "https://github.com/vudinhminh08/NLP-project-master-study.git"
REPO_BRANCH = "master"
PROJECT_DIR = "/kaggle/working/absa-project"

# Clone (hoặc pull nếu đã có)
if not os.path.exists(PROJECT_DIR):
    print(f"Cloning {REPO_URL} (branch: {REPO_BRANCH})...")
    !git clone --branch {REPO_BRANCH} --depth=1 {REPO_URL} {PROJECT_DIR}
    print("✅ Clone xong")
else:
    print(f"Repo đã tồn tại tại {PROJECT_DIR} — pulling latest...")
    !cd {PROJECT_DIR} && git pull origin {REPO_BRANCH}

# Chuyển vào thư mục project
os.chdir(PROJECT_DIR)
print(f"Working dir: {os.getcwd()}")

# Tạo thư mục cần thiết
for d in ["data", "outputs/models", "outputs/results", "outputs/eda"]:
    os.makedirs(d, exist_ok=True)

# Thêm Python paths
# code/week1 và code/week2 là compatibility aliases trỏ tới source hiện tại.
sys.path.insert(0, "code/week1")
sys.path.insert(0, "code/week2")

# Kiểm tra cấu trúc repo
print("\nCấu trúc repo:")
!ls -la
!ls -la code/week1 code/week2

Cloning https://github.com/vudinhminh08/NLP-project-master-study.git (branch: master)...
Cloning into '/kaggle/working/absa-project'...
remote: Enumerating objects: 94, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (87/87), done.
remote: Total 94 (delta 6), reused 77 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (94/94), 3.12 MiB | 20.08 MiB/s, done.
Resolving deltas: 100% (6/6), done.
✅ Clone xong
Working dir: /kaggle/working/absa-project

Cấu trúc repo:
total 52
drwxr-xr-x 9 root root 4096 Mar 30 16:14  .
drwxr-xr-x 3 root root 4096 Mar 30 16:14  ..
drwxr-xr-x 7 root root 4096 Mar 30 16:14  code
drwxr-xr-x 2 root root 4096 Mar 30 16:14  data
drwxr-xr-x 8 root root 4096 Mar 30 16:14  .git
-rw-r--r-- 1 root root  505 Mar 30 16:14  .gitignore
-rw-r--r-- 1 root root 3443 Mar 30 16:14  GUIDE_RUN_WEEK2.md
drwxr-xr-x 2 root root 4096 Mar 30 16:14  notebooks
drwxr-xr-x 5 root root 4096 Mar 30 16:14  outputs
-rw-r--r-- 1 root root 1238 Mar 3

In [3]:
# ============================================================
# Cell 3 — Download dataset VLSP 2018
# ============================================================
import pandas as pd, os

if not os.path.exists("data/train.csv"):
    print("Downloading VLSP 2018 Hotel dataset...")
    !git clone https://github.com/ds4v/absa-vlsp-2018.git /tmp/ds4v --depth=1
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/train.csv data/
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/dev.csv data/
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/test.csv data/
    print("✅ Data downloaded")
else:
    print("✅ Data đã tồn tại")

for split in ["train", "dev", "test"]:
    df = pd.read_csv(f"data/{split}.csv")
    print(f"  {split}: {len(df)} rows × {df.shape[1]} cols")

✅ Data đã tồn tại
  train: 3000 rows × 35 cols
  dev: 2000 rows × 35 cols
  test: 600 rows × 35 cols


In [4]:
# ============================================================
# Cell 4 — Preprocessing
# ============================================================
import pandas as pd, os

FORCE_REPROCESS = False

if (not FORCE_REPROCESS) and os.path.exists("data/train_preprocessed.csv"):
    print("✅ Cache đã có (data/*_preprocessed.csv)")
    s = pd.read_csv("data/train_preprocessed.csv").iloc[0]
    print(f"  Original : {s['Review'][:80]}")
    print(f"  Processed: {str(s.get('processed_review', 'N/A'))[:80]}")
else:
    print("Chưa có cache — chạy preprocessing với VnCoreNLP...")
    from step3_preprocessing import preprocess_dataframe, VnCoreNLPSegmenter
    import py_vncorenlp
    vncorenlp_dir = os.path.join(os.getcwd(), 'vncorenlp')
    if not os.path.exists(os.path.join(vncorenlp_dir, 'models', 'wordsegmenter', 'wordsegmenter.rdr')):
        print('Downloading VnCoreNLP models...')
        py_vncorenlp.download_model(save_dir=vncorenlp_dir)
    segmenter = VnCoreNLPSegmenter(vncorenlp_dir=vncorenlp_dir, use_fallback=False)
    for split in ["train", "dev", "test"]:
        df = pd.read_csv(f"data/{split}.csv")
        preprocess_dataframe(df, segmenter=segmenter,
                             cache_path=f"data/{split}_preprocessed.csv")
        print(f"  ✅ {split}: {len(df)} rows processed")
    segmenter.close()
    print("✅ Preprocessing hoàn tất")

✅ Cache đã có (data/*_preprocessed.csv)
  Original : Rộng rãi KS mới nhưng rất vắng. Các dịch vụ chất lượng chưa cao và thiếu.
  Processed: Rộng_rãi khách_sạn mới nhưng rất vắng . Các dịch_vụ chất_lượng chưa cao và thiếu


In [ ]:
# ============================================================
# Cell 5 — Verify config hiện tại
# ============================================================
import json, os

enc_cfg = json.load(open('outputs/eda/encoder_config.json'))
print('=== Encoder Config ===')
for k, v in enc_cfg.items():
    print(f'  {k}: {v}')

cw = json.load(open('outputs/eda/class_weights.json'))
print('
=== Global Class Weights ===')
label_map = {'0': 'absent', '1': 'positive', '2': 'negative', '3': 'neutral'}
for cls, w in cw['global_weights'].items():
    note = ' ← clip về 10.0' if float(w) > 10 else ''
    print(f"  {label_map.get(cls,cls):12s}: {float(w):.1f}x{note}")

from utils.constants import TRAIN_CONFIG, ZERO_TRAIN_ASPECTS, PHOBERT_MODEL_NAME

print('
=== Train Config hiện tại ===')
for k, v in TRAIN_CONFIG.items():
    print(f'  {k}: {v}')
print(f'  model: {PHOBERT_MODEL_NAME}')

# Verify các điều kiện bắt buộc để rerun với source hiện tại.
# Không assert learning_rate=1e-4 / optimizer=Adam theo notebook cũ vì source hiện tại dùng AdamW + LR 3e-5.
assert PHOBERT_MODEL_NAME == 'vinai/phobert-base-v2', f"Model sai: {PHOBERT_MODEL_NAME}"
assert ZERO_TRAIN_ASPECTS == ['ROOM_AMENITIES#PRICES']
assert TRAIN_CONFIG['encoder_option'] == 'cls_only', f"Encoder mặc định sai: {TRAIN_CONFIG['encoder_option']}"
assert TRAIN_CONFIG['optimizer'] in {'AdamW', 'Adam'}
print(f'
  ZERO_TRAIN_ASPECTS: {ZERO_TRAIN_ASPECTS}')
print('
✅ Config hiện tại hợp lệ để rerun PhoBERT')


=== Encoder Config ===
  recommended_max_seq_len: 256
  p99_word_count: 243
  encoder_option: concat_4_layers
  encoder_hidden_size: 3072
  note: PhoBERT max_position_embeddings=258, safe max=256

=== Global Class Weights ===
  absent      : 1.0x
  positive    : 8.6x
  negative    : 28.0x ← clip về 10.0
  neutral     : 154.5x ← clip về 10.0

=== Train Config (ds4v sync) ===
  learning_rate: 0.0001
  warmup_ratio: 0.15
  batch_size: 16
  grad_accumulation_steps: 1
  max_epochs: 20
  early_stop_patience: 7
  dropout: 0.2
  optimizer: Adam
  scheduler: cosine_warmup
  seed: 42
  max_seq_len: 256
  weight_clip: 10.0
  encoder_option: concat_4_layers
  max_grad_norm: 1.0
  model: vinai/phobert-base-v2

  ZERO_TRAIN_ASPECTS: ['ROOM_AMENITIES#PRICES']

✅ Config OK — ds4v sync verified


In [6]:
# ============================================================
# Cell 6 — TRAIN: concat_4_layers (config anti-OOM đã set ở Cell 5)
# ============================================================
import torch
torch.cuda.empty_cache()
print(f"VRAM free before training: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0))/1e9:.1f} GB")

from run_experiment import main

test_metrics = main(encoder_option='concat_4_layers', use_amp=True)

print('\n' + '='*55)
print('MAIN RUN RESULTS')
print(f"  ACD F1:      {test_metrics['macro_acd_f1']:.4f}  (SOTA: 0.8255)")
print(f"  SPC F1:      {test_metrics['macro_spc_f1']:.4f}")
print(f"  Combined F1: {test_metrics['macro_combined_f1']:.4f}  (SOTA: 0.7732)")
print('='*55)


VRAM free before training: 15.6 GB
[Device] GPU: Tesla T4

TUẦN 2 — PhoBERT Multi-task ABSA
  encoder:    concat_4_layers
  seq_len:    256
  batch:      16 × 1 = 16 effective
  lr:         0.0001
  weight_clip:10.0
  amp:        ON

[Tokenizer] Loading vinai/phobert-base-v2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

...


  Dev — Epoch 13
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* FOOD&DRINKS#MISCELLANEOUS          0.0000    0.0000    0.0000    0.0000    11
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
* FACILITIES#MISCELLANEOUS           0.1333    0.1250    0.1429    0.3333    7
FACILITIES#COMFORT                   0.3542    0.6296    0.2464    0.2636    69
ROOMS#QUALITY                        0.3750    0.3000    0.5000    0.4333    6
HOTEL#MISCELLANEOUS                  0.3902    0.6667    0.2759    0.2187    145
FACILITIES#GENERAL                   0.4341    0.4179    0.4516    0.2171    62
FOOD&DRINKS#PRICES                   0.5000    0.5185    0.4828    0.3844    29
FACILITIES#PRI

Train:   0%|          | 0/188 [00:00<?, ?it/s]/kaggle/working/absa-project/code/week2/train.py:138: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



  Dev — Epoch 14
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* FACILITIES#MISCELLANEOUS           0.0000    0.0000    0.0000    0.0000    7
* FOOD&DRINKS#MISCELLANEOUS          0.0000    0.0000    0.0000    0.0000    11
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
HOTEL#MISCELLANEOUS                  0.3315    0.8333    0.2069    0.1747    145
ROOMS#QUALITY                        0.3333    0.2500    0.5000    0.4333    6
FACILITIES#COMFORT                   0.4074    0.5641    0.3188    0.2875    69
FACILITIES#GENERAL                   0.4677    0.4677    0.4677    0.2222    62
FOOD&DRINKS#PRICES                   0.4898    0.6000    0.4138    0.3367    29
FACILITIES#PRI


  Final — DEV
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* FOOD&DRINKS#MISCELLANEOUS          0.0000    0.0000    0.0000    0.0000    11
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
* FACILITIES#MISCELLANEOUS           0.1111    0.0909    0.1429    0.3333    7
ROOMS#QUALITY                        0.1538    0.0870    0.6667    0.5333    6
HOTEL#MISCELLANEOUS                  0.3167    0.4605    0.2414    0.1796    145
FACILITIES#COMFORT                   0.3876    0.4167    0.3623    0.3527    69
FACILITIES#CLEANLINESS               0.4118    0.3111    0.6087    0.3360    23
FACILITIES#GENERAL                   0.4265    0.3919    0.4677    0.2222    62
ROOM_AMENITIES#QU


  Final — TEST
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* FACILITIES#MISCELLANEOUS           0.0000    0.0000    0.0000    0.0000    8
* FOOD&DRINKS#MISCELLANEOUS          0.0000    0.0000    0.0000    0.0000    3
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    4
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    1
FACILITIES#CLEANLINESS               0.2400    0.1500    0.6000    0.1667    5
HOTEL#MISCELLANEOUS                  0.2857    0.4667    0.2059    0.1528    68
ROOMS#QUALITY                        0.3784    0.2593    0.7000    0.5079    10
FACILITIES#COMFORT                   0.4231    0.4231    0.4231    0.3175    26
FACILITIES#GENERAL                   0.4590    0.3500    0.6667    0.5833    21
FACILITIES#PRICES 

In [7]:
# ============================================================
# Cell 7 — Ablation: cls_only
# ============================================================
# Chứng minh kỹ thuật concat 4 layers có đóng góp thực sự
# (so sánh 3072 dim vs 768 dim trong báo cáo)
# Chạy SAU khi Cell 6 đã hoàn tất

import torch
torch.cuda.empty_cache()

from run_experiment import main

test_metrics_cls = main(encoder_option="cls_only", use_amp=True)

print("\n" + "="*55)
print("ABLATION SUMMARY")
print(f"  concat_4_layers: {test_metrics['macro_combined_f1']:.4f}  ← SOTA architecture")
print(f"  cls_only:        {test_metrics_cls['macro_combined_f1']:.4f}")
gain = test_metrics['macro_combined_f1'] - test_metrics_cls['macro_combined_f1']
print(f"  Gain từ concat:  {gain*100:+.2f}%  {'✅ concat tốt hơn' if gain > 0 else '⚠️ cls_only bằng hoặc tốt hơn'}")
print("="*55)

[Device] GPU: Tesla T4

TUẦN 2 — PhoBERT Multi-task ABSA
  encoder:    cls_only
  seq_len:    256
  batch:      16 × 1 = 16 effective
  lr:         0.0001
  weight_clip:10.0
  amp:        ON

[Tokenizer] Loading vinai/phobert-base-v2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[Tokenizer] Loaded ✓

[Data] Creating DataLoaders...
[DataLoader] train: 3000 samples, 188 batches
[DataLoader] dev: 2000 samples, 125 batches
[DataLoader] test: 600 samples, 38 batches
[Weights] 34 aspects loaded, 77 weight values clipped at 10.0

[Model] Building ABSAPhoBERT (cls_only)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/kaggle/working/absa-project/code/week2/train.py:244: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if (use_amp and AMP_AVAILABLE and device.type == "cuda") else None



[Model] 135,102,856 trainable parameters
[Scheduler] Total=3760 optimizer steps, Warmup=564
[AMP] Mixed precision: ON ✓
[Config] encoder=cls_only, seq_len=256, batch=16×1=16 (effective)

────────────────────────────────────────────────────────────
Epoch 1/20


...


  Dev — Epoch 19
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* FOOD&DRINKS#MISCELLANEOUS          0.0000    0.0000    0.0000    0.0000    11
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
ROOMS#QUALITY                        0.2000    0.1429    0.3333    0.3333    6
* FACILITIES#MISCELLANEOUS           0.2222    0.5000    0.1429    0.0952    7
HOTEL#MISCELLANEOUS                  0.3196    0.6327    0.2138    0.1633    145
FACILITIES#COMFORT                   0.3818    0.5122    0.3043    0.3124    69
FACILITIES#GENERAL                   0.4638    0.4211    0.5161    0.2370    62
ROOM_AMENITIES#QUALITY               0.4726    0.3644    0.6721    0.5310    122
FOOD&DRINKS#P

Train:   0%|          | 0/188 [00:00<?, ?it/s]/kaggle/working/absa-project/code/week2/train.py:138: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



  Dev — Epoch 20
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* FOOD&DRINKS#MISCELLANEOUS          0.0000    0.0000    0.0000    0.0000    11
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
ROOMS#QUALITY                        0.2000    0.1429    0.3333    0.3333    6
* FACILITIES#MISCELLANEOUS           0.2222    0.5000    0.1429    0.0952    7
HOTEL#MISCELLANEOUS                  0.3212    0.6458    0.2138    0.1633    145
FACILITIES#COMFORT                   0.3818    0.5122    0.3043    0.3124    69
FACILITIES#GENERAL                   0.4638    0.4211    0.5161    0.2370    62
ROOM_AMENITIES#QUALITY               0.4699    0.3612    0.6721    0.5310    122
FOOD&DRINKS#P


  Final — DEV
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* FOOD&DRINKS#MISCELLANEOUS          0.0000    0.0000    0.0000    0.0000    11
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
ROOMS#QUALITY                        0.1739    0.1176    0.3333    0.3333    6
HOTEL#MISCELLANEOUS                  0.3061    0.5882    0.2069    0.1592    145
FACILITIES#COMFORT                   0.3860    0.4889    0.3188    0.3233    69
* FACILITIES#MISCELLANEOUS           0.4000    0.6667    0.2857    0.4286    7
FACILITIES#GENERAL                   0.4627    0.4306    0.5000    0.2322    62
ROOM_AMENITIES#QUALITY               0.4670    0.3382    0.7541    0.5643    122
FACILITIES#PRICE


  Final — TEST
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* FACILITIES#MISCELLANEOUS           0.0000    0.0000    0.0000    0.0000    8
* FOOD&DRINKS#MISCELLANEOUS          0.0000    0.0000    0.0000    0.0000    3
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    4
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    1
HOTEL#MISCELLANEOUS                  0.3146    0.6667    0.2059    0.1806    68
FACILITIES#CLEANLINESS               0.4000    0.4000    0.4000    0.2667    5
FACILITIES#GENERAL                   0.4151    0.3438    0.5238    0.2299    21
FACILITIES#COMFORT                   0.4878    0.6667    0.3846    0.1905    26
ROOM_AMENITIES#QUALITY               0.5000    0.3853    0.7119    0.4730    59
ROOMS#QUALITY     

In [8]:
# ============================================================
# Cell 8 — Learning Curve (BẮT BUỘC cho báo cáo)
# ============================================================
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

history = json.load(open("outputs/results/training_history.json"))
best_ep = history["best_epoch"]
epochs  = range(1, len(history["train_loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("PhoBERT concat_4_layers — Learning Curve (ABSA VLSP 2018)", fontsize=13)

# --- Loss ---
ax1.plot(epochs, history["train_loss"], "o-", c="crimson",   lw=2, label="Train Loss")
ax1.plot(epochs, history["dev_loss"],   "o-", c="steelblue", lw=2, label="Dev Loss")
ax1.axvline(best_ep, c="green", ls="--", alpha=0.7, label=f"Best (epoch {best_ep})")
ax1.set(title="Loss", xlabel="Epoch", ylabel="Cross-Entropy Loss")
ax1.legend(); ax1.grid(alpha=0.3)

# --- F1 ---
ax2.plot(epochs, history["dev_acd_f1"],      "s-", c="darkorange", lw=2, label="Dev ACD F1")
ax2.plot(epochs, history["dev_spc_f1"],      "^-", c="purple",     lw=2, label="Dev SPC F1")
ax2.plot(epochs, history["dev_combined_f1"], "o-", c="green",      lw=2.5, label="Dev Combined F1")
ax2.axvline(best_ep, c="green", ls="--", alpha=0.7, label=f"Best (epoch {best_ep})")
ax2.axhline(0.7732,  c="red",   ls=":",  alpha=0.5, label="SOTA Combined 0.7732")
ax2.set(title="F1 Score", xlabel="Epoch", ylabel="Macro F1")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/eda/learning_curve.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Phân tích cho báo cáo ---
print(f"\n📊 Phân tích learning curve (cho báo cáo):")
print(f"  Best epoch:           {best_ep} / {len(history['train_loss'])}")
print(f"  Best Combined F1:     {history['best_combined_f1']:.4f}")

if len(history["train_loss"]) > best_ep:
    tloss_at   = history["train_loss"][best_ep - 1]
    tloss_after = history["train_loss"][best_ep]
    dloss_at   = history["dev_loss"][best_ep - 1]
    dloss_after = history["dev_loss"][best_ep]
    print(f"  Train loss epoch {best_ep}: {tloss_at:.4f} → epoch {best_ep+1}: {tloss_after:.4f} (tiếp tục giảm)")
    print(f"  Dev loss epoch {best_ep}:   {dloss_at:.4f} → epoch {best_ep+1}: {dloss_after:.4f} (tăng = overfit)")
    print(f"  → Early stopping đúng: dev F1 không cải thiện {history['config']['early_stop_patience']} epoch liên tiếp")

print(f"\n  Saved: outputs/eda/learning_curve.png")


📊 Phân tích learning curve (cho báo cáo):
  Best epoch:           7 / 14
  Best Combined F1:     0.5134
  Train loss epoch 7: 0.2199 → epoch 8: 0.1738 (tiếp tục giảm)
  Dev loss epoch 7:   0.3053 → epoch 8: 0.3105 (tăng = overfit)
  → Early stopping đúng: dev F1 không cải thiện 7 epoch liên tiếp

  Saved: outputs/eda/learning_curve.png


In [9]:
# ============================================================
# Cell 9 — Summary Report & Copy kết quả
# ============================================================
import os, json, shutil

# In summary report
report = "outputs/results/phobert_summary.md"
if os.path.exists(report):
    print(open(report, encoding="utf-8").read())
else:
    print("Chưa có report — đảm bảo Cell 6 đã chạy xong")

# Liệt kê tất cả files kết quả
print("\n=== Kết quả đã tạo ===")
result_files = []
for root, dirs, files in os.walk("outputs"):
    for f in files:
        if not f.endswith(".DS_Store"):
            path = os.path.join(root, f)
            size = os.path.getsize(path)
            result_files.append(path)
            print(f"  {path} ({size/1024:.1f} KB)")

# Tạo zip để download
print("\nTạo zip...")
shutil.make_archive("/kaggle/working/phobert_results", "zip", "outputs")
print("✅ Zip: /kaggle/working/phobert_results.zip")
print("   → Kaggle: Output panel → Download")

# Copy summary để dễ copy-paste
print("\n=== Số liệu quan trọng để điền vào báo cáo ===")
if os.path.exists("outputs/results/phobert_test_metrics.json"):
    m = json.load(open("outputs/results/phobert_test_metrics.json"))
    print(f"__PHOBERT_CONCAT_ACD_F1__      = {m['macro_acd_f1']:.4f}")
    print(f"__PHOBERT_CONCAT_SPC_F1__      = {m['macro_spc_f1']:.4f}")
    print(f"__PHOBERT_CONCAT_COMBINED_F1__ = {m['macro_combined_f1']:.4f}")
if os.path.exists("outputs/results_cls_only/phobert_test_metrics.json"):
    m2 = json.load(open("outputs/results_cls_only/phobert_test_metrics.json"))
    print(f"__PHOBERT_CLS_ONLY_COMBINED_F1__ = {m2['macro_combined_f1']:.4f}")

# Kết quả Tuần 2 — PhoBERT Multi-task

## Config thực tế
| Tham số | Giá trị |
|---------|---------|
| Encoder | concat_4_layers |
| MAX_SEQ_LEN | 256 |
| Batch size | 16 × 1 = 16 (effective) |
| Learning rate | 0.0001 |
| Warmup | 15% steps |
| Weight clip | 10.0 |
| Best epoch | 7 / 14 |

## Kết quả

| Split | ACD F1 | SPC F1 | Combined F1 |
|-------|--------|--------|-------------|
| Dev   | 0.5624 | 0.4643 | 0.5134 |
| **Test**  | **0.5836** | **0.4658** | **0.5247** |
| SOTA (Huynh 2022) | 0.8255 | — | 0.7732 |

## Phân tích Gap so với SOTA

- **ACD F1 gap:** 0.2419 (24.2%)
- **Combined F1 gap:** 0.2485 (24.8%)

### Nguyên nhân gap (phân tích):
1. **underthesea vs VnCoreNLP:** Dùng underthesea làm fallback → ~1-2% F1 loss
2. **ROOM_AMENITIES#PRICES:** 0 training samples → ACD F1 = 0 cho aspect này
3. **Neutral cực hiếm (weight=154→clip=10):** SPC F1 cho neutral thấp
4. **Dataset nhỏ (3000 train):** SOTA có thể dùng data augmentation
5. **Single run:** Chưa ensemble nhiều seeds

##